# 03 - Estensione ad altri pazienti virtuali (adult#001, #002, #003)

**Obiettivo.** Ripetere il confronto su altri pazienti virtuali con le stesse condizioni del notebook 02 (stessi guadagni
e parametri, pasto non annunciato alle 12:00, stesso seme). I controllori **non** vengono ritarati sul singolo paziente:
cambiano solo le grandezze che dipendono dal basale del paziente (limiti dell'integrale e tetto IOB).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import aid_sim as A, notebook_utils as U
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 110)
SEED = 1                                  # seme del rumore del sensore (uguale per tutti i controllori)
COL = {"PID senza AW": "tab:gray", "PID": "tab:red", "PID cons.": "tab:orange",
       "PID+PLGS": "tab:green", "PID+IOB": "tab:purple", "PID+IOB+PLGS": "tab:blue"}
print("Versioni delle librerie usate:"); display(U.library_versions())

In [ ]:
def finestra(df):
    """Solo la finestra di analisi (08:00-24:00): l'assestamento 00:00-08:00 non entra né nei grafici né nelle metriche."""
    return df[df.t_h >= A.WARMUP_H]

def decora(ax, meal_h=A.MEAL_CLOCK_H, banda=True):
    """Linea del pasto e griglia; banda 70-180 mg/dL solo sui pannelli di glicemia."""
    if banda:
        ax.axhspan(70, 180, color="green", alpha=0.12)
    ax.axvline(meal_h, color="k", ls=":", lw=1.2); ax.grid(ls=":", alpha=0.5)

def tabella(risultati, chiavi):
    """risultati: dict {chiave: DataFrame}; chiavi: nomi delle colonne-indice per la tabella."""
    rows = []
    for k, df in risultati.items():
        k = k if isinstance(k, tuple) else (k,)
        rows.append({**dict(zip(chiavi, k)), **{m: round(float(v), 1) for m, v in A.metrics(df).items()}})
    return pd.DataFrame(rows)

## 1. Configurazione
### 1.1 Pazienti (parametri di simglucose)

In [ ]:
import os, simglucose
PAZ = ["adult#001", "adult#002", "adult#003"]
q = pd.read_csv(os.path.join(os.path.dirname(simglucose.__file__), "params", "Quest.csv")).set_index("Name")
rows = []
for p in PAZ:
    ic = U.initial_conditions(p, SEED)
    rows.append({"Paziente": p, "BW [kg]": round(ic["BW"], 1), "Gb [mg/dL]": round(ic["Gb"], 1), "u_b [U/h]": round(ic["ub"] * 60, 2),
                 "CR [g/U]": q.loc[p, "CR"], "CF [mg/dL/U]": round(q.loc[p, "CF"], 1), "TDI [U/die]": round(q.loc[p, "TDI"], 1),
                 "kabs [1/min]": round(ic["kabs"], 4)})
display(pd.DataFrame(rows))

### 1.2 Scenario e controllori

In [ ]:
display(U.scenario_table("adult#001", "0, 60 o 90", SEED).iloc[6:])
NOMI = ["PID senza AW", "PID", "PID cons.", "PID+IOB+PLGS"]
for p in PAZ:
    ub_p = U.initial_conditions(p, SEED)["ub"]
    t = U.controller_table(A.CFG["PID+IOB+PLGS"], ub_p).set_index("Parametro").loc[["I_min, I_max (derivati)", "Ki*I_min, Ki*I_max", "iob_max (derivato)"]]
    print(p, "- grandezze che dipendono dal basale (PID+IOB+PLGS):"); display(t)

### Unità di misura delle variabili salvate
| Variabile | Unità | Significato |
|---|---|---|
| `t_h` | h | ora del giorno (la simulazione parte alle 00:00) |
| `CGM` | mg/dL | misura del sensore, con rumore: è l'unica grandezza vista dal controllore |
| `BG` | mg/dL | glicemia plasmatica del modello UVa/Padova (Gp/Vg), usata per le metriche |
| `Gsub` | mg/dL | glicemia sottocutanea senza rumore (`info['bg']` di simglucose) |
| `Insulin_Uh` | U/h | velocità di infusione comandata |
| `IOB` | U | insulina attiva netta stimata dal controllore (eccesso rispetto al basale) |
| `IOB_vero` | U | insulina sottocutanea netta del modello (Isc1 + Isc2 meno l'equilibrio basale) |
| `Vincolo` | - | vincolo attivo nel passo: `iob`, `plgs`, `sat_alta`, `sat_bassa` o vuoto |

Metriche: TIR, TAR, TBR in % del tempo nella finestra 08:00-24:00; `TIR_post6h` nelle 6 h dopo il pasto;
`TBR_tardivo` dalle 3 h dopo il pasto alle 24:00; picco e nadir in mg/dL; `insulina_tot` in U nella finestra.

## 2. Verifica a digiuno e pasti da 60 e 90 g

In [ ]:
CARICHI = [0.0, 60.0, 90.0]
R = {(p, int(c), n): A.simulate(p, c, A.CFG[n], seed=SEED)[0] for p in PAZ for c in CARICHI for n in NOMI}
print(len(R), "simulazioni completate")

## 3. Grafici (60 g): glicemia e IOB per paziente

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(13, 9), sharex=True)
for i, p in enumerate(PAZ):
    ub_p = U.initial_conditions(p, SEED)["ub"]
    for n in NOMI:
        d = finestra(R[(p, 60, n)])
        ax[i, 0].plot(d.t_h, d.BG, color=COL[n], label=n); ax[i, 1].plot(d.t_h, d.IOB, color=COL[n])
    ax[i, 1].axhline(1.5 * ub_p * 60, color="gray", ls="-.")
    decora(ax[i, 0]); decora(ax[i, 1], banda=False)
    ax[i, 0].set_ylabel(f"{p}\nBG plasmatica [mg/dL]"); ax[i, 1].set_ylabel("IOB stimato [U]")
ax[0, 0].legend(fontsize=8); ax[2, 0].set_xlabel("Ora del giorno [h]"); ax[2, 1].set_xlabel("Ora del giorno [h]")
ax[2, 0].set_xlim(A.WARMUP_H, A.ANALYSIS_END_H); plt.tight_layout(); plt.show()

## 4. Indicatori clinici

In [ ]:
T = tabella(R, ["Paziente", "CHO [g]", "Controllore"])
for c in [0, 60, 90]:
    print(f"Pasto da {c} g:" if c else "A digiuno (0 g):")
    display(T[T["CHO [g]"] == c].drop(columns="CHO [g]").reset_index(drop=True))

## 5. Lettura dei risultati (calcolata dalla tabella)

In [ ]:
for p in PAZ:
    for c in [60, 90]:
        a = T[(T.Paziente == p) & (T["CHO [g]"] == c) & (T.Controllore == "PID")].iloc[0]
        b = T[(T.Paziente == p) & (T["CHO [g]"] == c) & (T.Controllore == "PID+IOB+PLGS")].iloc[0]
        print(f"{p} {c} g: TBR {a.TBR:4.1f} -> {b.TBR:4.1f} % | TBR<54 {a['TBR<54']:4.1f} -> {b['TBR<54']:4.1f} % | "
              f"TIR {a.TIR:5.1f} -> {b.TIR:5.1f} % | picco {a.picco:4.0f} -> {b.picco:4.0f} mg/dL")

**Limiti.** Guadagni e soglie non sono ritarati sul singolo paziente e non sono stati ottimizzati: i risultati mostrano il
comportamento con parametri fissi, non il miglior risultato ottenibile per ciascun soggetto.